In this notebook you will learn about core components of LSTM
- Cell state and hidden state with their functionalities
- Gates: input, forget, output, equations behind them


To start this lesson **all students must be aware** of the following concepts:

- Sequential Networks
- Basic of LSTM
- Workings of Recurrent Neural Networks
- Activation Functions like:  Tanh, Sigmoid

# **Gated Memory Cell**

LSTMs are similar to standard recurrent neural networks (RNNs), but instead of using simple recurrent nodes, each unit is replaced by a memory cell. A memory cell has an internal state that works like a storage unit, helping the network remember information for a long time. This internal state is connected to itself with a fixed weight, which allows information (and gradients during training) to pass through many time steps without quickly fading away (vanishing) or growing uncontrollably (exploding).

To control this information, memory cells use special gates. These gates act like switches that decide what happens to the information at each step. The input gate controls whether new information should be added to the memory, the forget gate decides whether some old information should be erased, and the output gate decides what part of the memory should be sent out as the cell’s output.

A key idea in LSTMs is the cell state, which you can imagine as a conveyor belt running through the entire sequence. Information can travel along this conveyor belt from start to end with only small changes made by the gates. This makes it easy for important information to flow forward through time, while still allowing the network to update or remove unnecessary details when needed.



# **Gated Hidden State**

The primary distinction between vanilla RNNs and LSTMs lies in the gating mechanism applied to the hidden state. Instead of updating the hidden state blindly at each step, LSTMs learn when to update and when to reset. The update control mechanism allows the model to decide the right moment for modifying the hidden state, while the reset control mechanism provides a way to clear it when necessary. These learned controls address the common shortcomings of standard RNNs. For instance, if the very first token in a sequence carries important information, the LSTM can learn not to overwrite it in later steps. Similarly, if certain observations in the sequence are irrelevant, the model can simply skip the update. Finally, if the hidden state accumulates noise over time, the reset mechanism allows the model to clear it and start fresh.

*Here is a single cell state C and  hidden state h at time stamp t.*
<div align="center">

[![lstm-cell.png](https://i.postimg.cc/ryBJ7tyX/lstm-cell.png)](https://postimg.cc/SjGCMRzD)

*Figure: A single cell of LSTM*
</div>


## **Input Gate, Forget Gate, and Output Gate**  


The LSTM does have the ability to remove or add information to the cell state, carefully regulated by structures called gates. Gates are a way to optionally let information through. They are composed out of a sigmoid neural net layer and a pointwise multiplication operation.


The data feeding into the LSTM gates are the **input at the current time step and the hidden state of the previous time step**, as illustrated in figure below, three fully connected layers with sigmoid activation functions compute the values of the input, forget, and output gates. As a result of the sigmoid activation, all values of the three gates are in the range of $(0,1).$ Additionally, we require an **input node**, typically computed with a **tanh activation** function. Intuitively, the input gate determines how much of the input node’s value should be added to the current memory cell internal state. The forget gate determines whether to keep the current value of the memory or flush it. And the output gate determines whether the memory cell should influence the output at the current time step.  


<div align="center">
  <img src="https://d2l.ai/_images/lstm-0.svg"
       alt="LSTM gate computations"
       width="600" height="400">
  <p><b>Fig. 1:</b> LSTM gate computations.  
     Source: <a href="https://d2l.ai/chapter_recurrent-modern/lstm.html" target="_blank">Dive into Deep Learning (Zhang et al., 2024)</a></p>
</div>


Mathematically, suppose that there are **h hidden units**, the batch size is **n**, and the number of inputs is **d**.  
Thus the **input** is
  \$[
  \mathbf{X}_t \in \mathbb{R}^{n \times d}
  \]$
  
  and hidden state of the previous time step is    \$[
  \mathbf{H}_{t-1} \in \mathbb{R}^{n \times h}
  \]$

## **Forget Gate:**

The first step in our **LSTM** is to decide what information we’re going to throw away from the cell state. This decision is made by a **sigmoid layer** called the *forget gate layer*.  

It looks at **h<sub>t−1</sub>** and **x<sub>t</sub>**, and outputs a number between **0** and **1** for each number in the cell state **C<sub>t−1</sub>**.  

- **1** → completely keep this  
- **0** → completely get rid of this

Let's go back to our example in previous notebook,  **"John lived in Nepal for over 15 years. He listens to Nepali songs. He reads Nepali books. He is fluent in ___."**


The **forget gate** decides which past information to keep and which to discard from the cell state.

- When the LSTM reads **"John lived in Nepal…"**, the forget gate assigns high importance to **“Nepal”** and related context (location, culture).  
- At the same time, it may **downweight** or discard less relevant details like **“15 years”**, since that doesn’t directly help predict the missing word.

<div align='center'>

[![forget-gate.png](https://i.postimg.cc/brdpcBqg/forget-gate.png)](https://postimg.cc/tn0LFDZn)

*figure: Forget Gate*
</div>

If “Nepal” is important, \$( f_t \approx 1 \)$ for that memory, so it stays in the cell state.



## **Input Gate and Candidate State:**

The next step is to decide what new information we’re going to store in the cell state.  
This has two parts: A **sigmoid layer** called the *input gate layer* decides which values we’ll update. And a **tanh layer** creates a vector of new candidate values, **Ĉ<sub>t</sub>**, that could be added to the state. In the next step, we’ll combine these two to create an update to the state.

$
i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)
$


<div align="center">

[![input-gate.png](https://i.postimg.cc/dtbvp6pR/input-gate.png)](https://postimg.cc/xc3Wbv0q)

*Figure: Input State*
</div>


### **Candidate State (Input Node)**

Next, we create a **vector of candidate values** that could be added to the state.  
A **tanh activation** is applied to keep the values between -1 and 1.

$
\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)
$


<div align="center">

[![Input-node.png](https://i.postimg.cc/YSk7q9ph/Input-node.png)](https://postimg.cc/hXC62KQB)

*Figure: Candidate State*
</div>



It’s now time to **update the old cell state**, \$( C_{t-1} \)$, into the new cell state \( C_t \).  

The previous steps (forget gate and input gate) already decided what to do — now we just need to actually perform the update.

We multiply the old state by \(f_t \), forgetting the things we decided to discard earlier.
Add \$( i_t \ast \tilde{C}_t \)$, which are the new candidate values, scaled by how much we decided to update each state value.


In LSTMs, the **input gate** $(\mathbf{I}_t)$  governs how much we take new data into account via $\tilde{\mathbf{C}}_t$, and the **forget gate** $\mathbf{F}_t$ addresses how much of the old cell internal state $\mathbf{C}_{t-1}$in $\mathbb{R}^{n \times h}$ we retain.  

Using the **Hadamard (elementwise) product operator** $\odot$, we arrive at the following update equation:  

$$
\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t
$$  

If the forget gate is always 1 and the input gate is always 0, the memory cell internal state $\mathbf{C}_{t-1}$ will remain constant forever, passing unchanged to each subsequent time step.  

However, input gates and forget gates give the model the **flexibility to learn** when to keep this value unchanged and when to perturb it in response to subsequent inputs.  

In practice, this design **alleviates the vanishing gradient problem**, resulting in models that are much easier to train, especially when facing datasets with long sequence lengths.  



In our example **input gate** decides what new information to add to the cell state.
 When reading: “He listens to **Nepali** songs” → The input gate strongly updates the memory with the idea that *Nepali* is a recurring theme. “He reads **Nepali** books” → The gate again reinforces the *Nepali* context.  

So the cell state accumulates the information:  

**John → Nepal → Nepali songs → Nepali books.**
---

**Notes:**  
 Broadcasting  is triggered during the summation. We use **sigmoid activation function**  maps values to the interval \((0,1)\).  

###  Intuition  
- **Input Gate** → how much of the new input to store  
- **Forget Gate** → how much of the old memory to keep  
- **Output Gate** → how much of the memory to output


### **Output gate and Hidden State**

Last, we need to define how to compute the output of the memory cell, i.e., the **hidden state**  $
\mathbf{H}_t \in \mathbb{R}^{n \times h}
$ as seen by other layers. This is where the **output gate** comes into play.  

In LSTMs, we first apply a **tanh** activation to the memory cell internal state and then apply another point-wise multiplication, this time with the output gate. This ensures that the values of $\mathbf{H}_t$ are always in the interval $(-1, 1)$:  

$
\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t)
$


Whenever the output gate is close to 1, we allow the memory cell internal state to impact the subsequent layers **unhindered**.  

- If the output gate is close to 0 → the current memory is **blocked** from impacting other layers of the network.  
- If the output gate is close to 1 → the memory is **fully passed through**.  

This means that a memory cell can **accumulate information across many time steps without impacting the rest of the network** (as long as the output gate remains near 0), and then suddenly release that information once the output gate flips to values near 1.  

 $$
 $$



<div align="center">

[![output-gate.png](https://i.postimg.cc/k4YPcvWb/output-gate.png)](https://postimg.cc/xJmZjHxf)

*Figure: Complete LSTM Cell*
</div>

In our example the **output gate** decides what part of the cell state should influence the next prediction (the blank). At the point **“He is fluent in ___”**, the LSTM looks at its memory, strong signals for **“Nepali”** (from books + songs + Nepal) and weak signals from irrelevant parts (like “15 years”).  

The output gate lets the **Nepali context** flow into the final hidden state → the model predicts **“Nepali”**.


# **Key Takeaways**

- The memory cell in an LSTM is more advanced than the simple hidden state in an RNN, as it allows long-term information retention.

- The Forget Gate, Input Gate, and Output Gate each play crucial roles in controlling the flow of information, determining what to keep, update, or discard.

- Both the cell state and the hidden state are updated at each time step, enabling LSTMs to effectively capture long-term dependencies in sequential data.

## ***Resources***

https://d2l.ai/chapter_recurrent-modern/lstm.html
https://colah.github.io/posts/2015-08-Understanding-LSTMs/